# Amsterdam Co-Accessibility with CBS Population Data

This notebook tells a simple story with **one POI type only: parks**.

The question is: **to what extent do children and older adults have access to the same parks?**

If a park is accessible to both groups, it can be seen as a stronger opportunity for social encounters in public space. If it is mostly accessible to only one group, that opportunity is weaker.


In [ ]:
import accessx as acx
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd
from pathlib import Path


## Setup


In [ ]:
epsg_ams = 28992
resolution = 9
buffer_m = 1000
max_cost = 15
cost_attr = "avg_time"

coacc_path = Path("../data/case_studies/coaccessibility/amsterdam")
area_path = coacc_path / "area"
population_path = coacc_path / "population"
street_network_path = coacc_path / "street_network"
pois_path = coacc_path / "pois"
results_path = coacc_path / "results"

for folder in [coacc_path, area_path, population_path, street_network_path, pois_path, results_path]:
    folder.mkdir(parents=True, exist_ok=True)

cbs_path = Path("../data/nl_cbs/cbs_vk100_2020_vol.gpkg")

population_columns = {
    "population": "aantal_inwoners",
    "children": "aantal_inwoners_0_tot_15_jaar",
    "elderly": "aantal_inwoners_65_jaar_en_ouder",
}

tags_library = {
    "leisure_park": {"leisure": ["park"]},
}


## 1. Load Amsterdam


In [ ]:
gdf_ams = ox.geocode_to_gdf("Amsterdam, Netherlands")
acx.save_gdf(gdf_ams, area_path / "aoi_amsterdam.geojson")
# gdf_ams = acx.read_gdf(area_path / "aoi_amsterdam.geojson")


## 2. Create Hexagons


In [ ]:
hex_ams = acx.make_hex_grid(gdf_ams, resolution=resolution, clip=True)
acx.save_gdf(hex_ams, area_path / "hexes_amsterdam.geojson")
# hex_ams = acx.read_gdf(area_path / "hexes_amsterdam.geojson")


## 3. Load the CBS grid and keep only Amsterdam


In [ ]:
ams_bbox = tuple(gdf_ams.to_crs(epsg_ams).total_bounds)
cbs_ams = gpd.read_file(cbs_path, bbox=ams_bbox)
cbs_ams = cbs_ams.to_crs(epsg_ams)
cbs_ams = gpd.overlay(
    cbs_ams[["geometry"] + list(population_columns.values())],
    gdf_ams.to_crs(epsg_ams)[["geometry"]],
    how="intersection",
    keep_geom_type=False,
)

for source_col in population_columns.values():
    cbs_ams[source_col] = pd.to_numeric(cbs_ams[source_col], errors="coerce")
    cbs_ams[source_col] = cbs_ams[source_col].where(cbs_ams[source_col] >= 0, 0.0)

cbs_ams = cbs_ams.rename(columns={source: target for target, source in population_columns.items()})
acx.save_gdf(cbs_ams, population_path / "cbs_amsterdam_clipped.geojson")
# cbs_ams = acx.read_gdf(population_path / "cbs_amsterdam_clipped.geojson")

cbs_ams[["population", "children", "elderly"]].sum()


## 4. Map the CBS population to the hexagons


In [ ]:
hex_pop_ams = acx.map_population_grid_to_hexes(
    hex_ams,
    cbs_ams,
    metric_crs=epsg_ams,
    population_cols=["population", "children", "elderly"],
)
acx.save_gdf(hex_pop_ams, population_path / "hexes_population_amsterdam_cbs.geojson")
# hex_pop_ams = acx.read_gdf(population_path / "hexes_population_amsterdam_cbs.geojson")

hex_pop_ams[["population", "children", "elderly"]].sum()


## 5. Build the walking network


In [ ]:
graph_ams = acx.build_network(
    AOI=gdf_ams,
    city_epsg=epsg_ams,
    buffer_m=buffer_m,
    network_type="walk",
    simplify=False,
    retain_all=True,
)

graph_ams = acx.add_time_cost_constant_speed(
    graph_ams,
    speed_kmh=4.5,
    cost_col=cost_attr,
)

acx.save_graph(
    graph_ams,
    out_dir=street_network_path,
    base_name="amsterdam_graph_with_cost",
    save_nodes=True,
    save_edges=True,
)

# graph_ams = acx.load_graph(
#     nodes_path=street_network_path / "amsterdam_graph_with_cost_nodes_OSM.geojson",
#     edges_path=street_network_path / "amsterdam_graph_with_cost_edges_OSM.geojson",
#     crs=epsg_ams,
# )


## 6. Collect parks


In [ ]:
pois_ams = acx.get_pois_osm(gdf_ams, tags_library=tags_library)
acx.save_gdf(pois_ams, pois_path / "pois_amsterdam_parks.geojson")
# pois_ams = acx.read_gdf(pois_path / "pois_amsterdam_parks.geojson")

pois_ams.head()


## 7. Compute co-accessibility for parks

For each park, we estimate how many people, children, and older adults can reach it within 15 minutes on foot.


In [ ]:
coacc_parks = acx.compute_co_accessibility(
    graph_ams,
    hex_pop_ams,
    pois_ams,
    max_cost=max_cost,
    cost_attr=cost_attr,
    population_groups=["population", "children", "elderly"],
    poi_id_col="id",
    category_col="category",
    keep_poi_cols=["name"],
    approach="cumulative",
)
acx.save_gdf(coacc_parks, results_path / "coaccessibility_parks_cumulative.geojson")
# coacc_parks = acx.read_gdf(results_path / "coaccessibility_parks_cumulative.geojson")

coacc_parks.head()


## 8. Total population around each park

This shows which parks are reachable to many people overall.


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
hex_ams.plot(ax=ax, color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
coacc_parks.plot(
    ax=ax,
    column="coacc_population",
    cmap="YlOrRd",
    legend=True,
    markersize=14,
)
ax.set_title("Amsterdam parks | accessible total population")
ax.axis("off")
plt.show()


## 9. Children around each park

This highlights which parks are more accessible to children.


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
hex_ams.plot(ax=ax, color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
coacc_parks.plot(
    ax=ax,
    column="coacc_children",
    cmap="Blues",
    legend=True,
    markersize=14,
)
ax.set_title("Amsterdam parks | accessible children")
ax.axis("off")
plt.show()


## 10. Older adults around each park

This highlights which parks are more accessible to older adults.


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
hex_ams.plot(ax=ax, color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
coacc_parks.plot(
    ax=ax,
    column="coacc_elderly",
    cmap="Greens",
    legend=True,
    markersize=14,
)
ax.set_title("Amsterdam parks | accessible older adults")
ax.axis("off")
plt.show()


## 11. Do children and older adults have access to the same parks?

To answer this directly, we build three simple indicators:

- `children_share`: the share of the accessible population that is children
- `elderly_share`: the share of the accessible population that is older adults
- `shared_access`: the minimum of the two counts

`shared_access` is the simplest way to show whether both groups meaningfully overlap on the same park. High values mean stronger opportunities for social encounters across age groups.


In [ ]:
park_story = coacc_parks.copy()

park_story["children_share"] = 0.0
park_story["elderly_share"] = 0.0
mask = park_story["coacc_population"] > 0
park_story.loc[mask, "children_share"] = (
    park_story.loc[mask, "coacc_children"]
    / park_story.loc[mask, "coacc_population"]
)
park_story.loc[mask, "elderly_share"] = (
    park_story.loc[mask, "coacc_elderly"]
    / park_story.loc[mask, "coacc_population"]
)
park_story["children_minus_elderly"] = (
    park_story["children_share"] - park_story["elderly_share"]
)
park_story["shared_access"] = park_story[["coacc_children", "coacc_elderly"]].min(axis=1)

park_story[[
    "name",
    "coacc_population",
    "coacc_children",
    "coacc_elderly",
    "children_share",
    "elderly_share",
    "shared_access",
]].head()


## 12. Shared access to the same parks

These are the parks where both children and older adults can reach the same place in larger numbers.


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
hex_ams.plot(ax=ax, color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
park_story.plot(
    ax=ax,
    column="shared_access",
    cmap="Purples",
    legend=True,
    markersize=14,
)
ax.set_title("Amsterdam parks | shared access by children and older adults")
ax.axis("off")
plt.show()


## 13. Which parks lean more toward children or older adults?

This does not show total volume. It shows which parks are relatively more child-oriented or more older-adult-oriented.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
hex_ams.plot(ax=axes[0], color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
park_story.plot(
    ax=axes[0],
    column="children_share",
    cmap="Blues",
    legend=True,
    markersize=14,
)
axes[0].set_title("Share of accessible children")
axes[0].axis("off")

hex_ams.plot(ax=axes[1], color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
park_story.plot(
    ax=axes[1],
    column="elderly_share",
    cmap="Greens",
    legend=True,
    markersize=14,
)
axes[1].set_title("Share of accessible older adults")
axes[1].axis("off")

hex_ams.plot(ax=axes[2], color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
park_story.plot(
    ax=axes[2],
    column="children_minus_elderly",
    cmap="coolwarm",
    legend=True,
    markersize=14,
)
axes[2].set_title("Children share minus older-adult share")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## 14. Which parks stand out the most?


In [ ]:
top_shared_parks = park_story[[
    "name",
    "coacc_population",
    "coacc_children",
    "coacc_elderly",
    "shared_access",
]].sort_values("shared_access", ascending=False).head(10)

top_shared_parks


In [ ]:
top_child_oriented_parks = park_story[[
    "name",
    "coacc_population",
    "coacc_children",
    "children_share",
]].sort_values("children_share", ascending=False).head(10)

top_child_oriented_parks


In [ ]:
top_elderly_oriented_parks = park_story[[
    "name",
    "coacc_population",
    "coacc_elderly",
    "elderly_share",
]].sort_values("elderly_share", ascending=False).head(10)

top_elderly_oriented_parks


## 15. Optional: Hansen version

If you want a distance-decay version instead of the cumulative threshold version, you can compute it here.


In [ ]:
# coacc_parks_hansen = acx.compute_co_accessibility(
#     graph_ams,
#     hex_pop_ams,
#     pois_ams,
#     max_cost=max_cost,
#     cost_attr=cost_attr,
#     population_groups=["population", "children", "elderly"],
#     poi_id_col="id",
#     category_col="category",
#     keep_poi_cols=["name"],
#     approach="hansen",
#     beta=0.15,
# )
# acx.save_gdf(coacc_parks_hansen, results_path / "coaccessibility_parks_hansen.geojson")
# coacc_parks_hansen.head()
